In [1]:
%pip install -q chess zstandard pandas pyarrow requests tqdm


[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import io
import re
import requests
import zstandard as zstd
import chess
import chess.pgn
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from pathlib import Path

MONTH = "2024-06"
URL = f"https://database.lichess.org/standard/lichess_db_standard_rated_{MONTH}.pgn.zst"
N_GAMES = 10000
REQUIRE_EVAL = True
MIN_PLIES = 20
MIN_ELO, MAX_ELO = 600, 2800
OUT_DIR = Path("data")
OUT_DIR.mkdir(exist_ok=True)

In [3]:
def stream_game_texts(url):
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        reader = zstd.ZstdDecompressor().stream_reader(r.raw, read_across_frames=True)
        text = io.TextIOWrapper(reader, encoding="utf-8", errors="ignore")
        buf = []
        for line in text:
            if line.startswith("[Event ") and buf:
                yield "".join(buf)
                buf = []
            buf.append(line)
        if buf:
            yield "".join(buf)


HEADER_RE = re.compile(r'^\[(\w+) "(.*)"\]$', re.M)


def quick_filter(txt):
    h = dict(HEADER_RE.findall(txt))
    event = h.get("Event", "")
    if not (event.startswith("Rated Blitz") or event.startswith("Rated Rapid")):
        return None
    if "BOT" in (h.get("WhiteTitle", ""), h.get("BlackTitle", "")):
        return None
    if h.get("Termination") not in ("Normal", "Time forfeit"):
        return None
    if h.get("Variant", "Standard") != "Standard":
        return None
    try:
        we, be = int(h["WhiteElo"]), int(h["BlackElo"])
    except (KeyError, ValueError):
        return None
    if not (MIN_ELO <= we <= MAX_ELO and MIN_ELO <= be <= MAX_ELO):
        return None
    if "[%clk" not in txt:
        return None
    if REQUIRE_EVAL and "[%eval" not in txt:
        return None
    return h

In [4]:
PIECE_VALUES = [(chess.KNIGHT, 3), (chess.BISHOP, 3), (chess.ROOK, 5), (chess.QUEEN, 9)]


def parse_game(txt, h):
    game = chess.pgn.read_game(io.StringIO(txt))
    if game is None or game.errors:
        return None
    base, inc = map(int, h["TimeControl"].split("+"))
    gid = h.get("Site", "").rsplit("/", 1)[-1]
    board = game.board()
    last_clock = {chess.WHITE: float(base), chess.BLACK: float(base)}
    prev_eval = None
    rows = []
    for ply, node in enumerate(game.mainline()):
        move = node.move
        mover = board.turn
        clk = node.clock()
        ev = node.eval()
        ev_cp = ev.white().score(mate_score=10000) if ev is not None else None
        npm = sum(len(board.pieces(pt, c)) * v for pt, v in PIECE_VALUES for c in (chess.WHITE, chess.BLACK))
        rows.append(dict(
            game_id=gid,
            ply=ply,
            move_number=ply // 2 + 1,
            side="white" if mover == chess.WHITE else "black",
            fen=board.fen(),
            move=move.uci(),
            piece=board.piece_at(move.from_square).symbol().upper(),
            is_capture=board.is_capture(move),
            is_check=board.gives_check(move),
            n_legal=board.legal_moves.count(),
            non_pawn_material=npm,
            elo_self=int(h["WhiteElo"] if mover else h["BlackElo"]),
            elo_oppo=int(h["BlackElo"] if mover else h["WhiteElo"]),
            clock_before=last_clock[mover],
            clock_after=clk,
            eval_before_cp=prev_eval,
            eval_after_cp=ev_cp,
            base_time=base,
            increment=inc,
            event=h["Event"],
            eco=h.get("ECO"),
            opening=h.get("Opening"),
            result=h["Result"],
            termination=h["Termination"],
        ))
        if clk is not None:
            last_clock[mover] = clk
        prev_eval = ev_cp
        board.push(move)
    if len(rows) < MIN_PLIES:
        return None
    return rows

In [5]:
games, rows = [], []
scanned = 0
pbar = tqdm(total=N_GAMES)
for txt in stream_game_texts(URL):
    scanned += 1
    h = quick_filter(txt)
    if h is None:
        continue
    r = parse_game(txt, h)
    if r is None:
        continue
    rows.extend(r)
    games.append(dict(
        game_id=r[0]["game_id"],
        white_elo=int(h["WhiteElo"]),
        black_elo=int(h["BlackElo"]),
        event=h["Event"],
        time_control=h["TimeControl"],
        eco=h.get("ECO"),
        opening=h.get("Opening"),
        result=h["Result"],
        termination=h["Termination"],
        n_plies=len(r),
    ))
    pbar.update(1)
    if len(games) >= N_GAMES:
        break
pbar.close()
print(f"scanned {scanned} | kept games {len(games)} | positions {len(rows)}")

  0%|          | 0/10000 [00:00<?, ?it/s]

scanned 146094 | kept games 10000 | positions 676579


In [6]:
moves = pd.DataFrame(rows)
games_df = pd.DataFrame(games).drop_duplicates("game_id")
moves = moves[moves["game_id"].isin(games_df["game_id"])]

bad_games = moves.loc[moves["clock_after"].isna(), "game_id"].unique()
moves = moves[~moves["game_id"].isin(bad_games)].copy()
games_df = games_df[~games_df["game_id"].isin(bad_games)].copy()

moves["eval_before_cp"] = pd.to_numeric(moves["eval_before_cp"])
moves["eval_after_cp"] = pd.to_numeric(moves["eval_after_cp"])

moves["time_control_type"] = np.where(moves["event"].str.startswith("Rated Blitz"), "blitz", "rapid")
moves["time_spent"] = (moves["clock_before"] - moves["clock_after"] + moves["increment"]).clip(lower=0)
moves["clock_frac"] = moves["clock_before"] / moves["base_time"]
moves["low_time"] = moves["clock_before"] < 30

sign = np.where(moves["side"] == "white", 1, -1)


def win_chance(cp):
    cp = np.clip(cp, -1000, 1000)
    return 2 / (1 + np.exp(-0.00368208 * cp)) - 1


moves["wc_before"] = win_chance(sign * moves["eval_before_cp"])
moves["wc_after"] = win_chance(sign * moves["eval_after_cp"])
moves["wc_drop"] = moves["wc_before"] - moves["wc_after"]
moves["move_quality"] = pd.cut(
    moves["wc_drop"],
    [-np.inf, 0.1, 0.2, 0.3, np.inf],
    labels=["good", "inaccuracy", "mistake", "blunder"],
    right=False,
).astype(str).replace("nan", pd.NA)

moves["phase"] = np.select(
    [moves["move_number"] <= 12, moves["non_pawn_material"] <= 26],
    ["opening", "endgame"],
    "middlegame",
)

moves["elo_band"] = pd.cut(moves["elo_self"], bins=list(range(600, 3000, 200)), right=False).astype(str)
moves["elo_diff"] = moves["elo_self"] - moves["elo_oppo"]
moves["forced"] = moves["n_legal"] == 1

moves = moves.reset_index(drop=True)
print(moves.shape, games_df.shape)

(676579, 36) (10000, 10)


In [7]:
moves.to_parquet(OUT_DIR / f"moves_{MONTH}.parquet", index=False)
games_df.to_parquet(OUT_DIR / f"games_{MONTH}.parquet", index=False)

In [8]:
print(moves["time_control_type"].value_counts(), "\n")
print(moves["phase"].value_counts(), "\n")
print(moves["move_quality"].value_counts(dropna=False), "\n")
print(moves["elo_band"].value_counts().sort_index(), "\n")
print(moves[["clock_before", "time_spent", "wc_drop", "n_legal"]].describe().T, "\n")
print("eval coverage:", moves["eval_after_cp"].notna().mean().round(3))
print("forced moves:", moves["forced"].mean().round(3))

time_control_type
blitz    445278
rapid    231301
Name: count, dtype: int64 

phase
middlegame    282626
opening       239791
endgame       154162
Name: count, dtype: int64 

move_quality
good          533599
inaccuracy     64039
blunder        39674
mistake        26273
<NA>           12994
Name: count, dtype: int64 

elo_band
[1000, 1200)     57751
[1200, 1400)     84221
[1400, 1600)    108459
[1600, 1800)    115731
[1800, 2000)    109440
[2000, 2200)     78647
[2200, 2400)     47641
[2400, 2600)     29160
[2600, 2800)      4902
[600, 800)        8371
[800, 1000)      32256
Name: count, dtype: int64 

                 count        mean         std       min    25%         50%  \
clock_before  676579.0  254.871549  199.097976  0.000000  115.0  180.000000   
time_spent    676579.0    7.163990   11.783292  0.000000    2.0    3.000000   
wc_drop       663585.0    0.070907    0.159383 -0.950895    0.0    0.015621   
n_legal       676579.0   30.017184   11.756787  1.000000   24.0   32.0000